# HOD fitting with HODDIES

<!-- Loads the halo catalog of one run into HODDIES (dev-v2: a `Base_catalogue` of the halo array, then
a `HOD` object), as `jobs/hod_mock.sh` does, and checks it with one mock. The fit goes in the
last section. -->

In [ ]:
import json
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
from halo_pipeline import require_peculiar_velocities  # halo catalogs made before the velocity fix are refused
sys.path.insert(0, str(hp.HODDIES_ROOT))
for module_name in list(sys.modules):
    if module_name == 'HODDIES' or module_name.startswith('HODDIES.'):
        del sys.modules[module_name]

import numpy as np
import matplotlib.pyplot as plt
import HODDIES
from HODDIES import HOD
from HODDIES.sim_loader import Base_catalogue
from abacus_reference_cosmology import SERIALIZABLE_PARAMETERS

print(f'Using HODDIES from {HODDIES.__file__}')

In [ ]:
from cosmoprimo.fiducial import Cosmology

abacus_cosmo = Cosmology(
    Omega0_m=SERIALIZABLE_PARAMETERS['Omega_m'],
    Omega_b=SERIALIZABLE_PARAMETERS['Omega_b'],
    h=SERIALIZABLE_PARAMETERS['h'],
    sigma8=SERIALIZABLE_PARAMETERS['sigma8'],
    n_s=SERIALIZABLE_PARAMETERS['n_s'],
    w0_fld=SERIALIZABLE_PARAMETERS['w0'],
    wa_fld=SERIALIZABLE_PARAMETERS['wa'],
    T_cmb=SERIALIZABLE_PARAMETERS['Tcmb0_K'],
    N_eff=SERIALIZABLE_PARAMETERS['Neff'],
    m_ncdm=[],
    engine='class',
)

In [ ]:
target, model, realization = 'Abacus_Small', 'LCDM', 's1234'
redshift = 0.500
param_file = hp.NOTEBOOK_DIR / f'HOD_parameter_file_optimizer.yaml'  # HODDIES parameter file
tracer = 'LRG'
nthreads = 32   # HODDIES threads (capped by numba's thread count); the mock of a seed depends on it
seed = 42       # HOD seed
fit_name = 'LRG_wp+xi_cmaes'   # results in hicola_data/hicola/<target>/<model>/<realization>/HiCOLA_<sim>_z<z>_HOD-dev-v2_fit_<fit_name>/

## Halo catalog

In [ ]:
hoddies_file = hp.hoddies_catalog_file(target, model, realization, redshift)
catalog_file = hp.halo_catalog_file(target, model, realization, redshift)
if not hoddies_file.exists():
    raise FileNotFoundError(f'Missing {hoddies_file}. Run jobs/halo_finder.sh on '
                            f'{hp.run_file(target, model, realization)} at z = {redshift} first.')

halos = np.load(hoddies_file)
halo_metadata = json.loads(Path(str(catalog_file) + '.json').read_text())
require_peculiar_velocities(halo_metadata, catalog_file)
boxsize = float(halo_metadata['boxsize'])  # box size of the run, recorded by the halo finder

halo_cosmology = halo_metadata.get('cosmology') or {}
mismatches = {key: (halo_cosmology[key], expected) for key, expected in SERIALIZABLE_PARAMETERS.items()
              if key in halo_cosmology and not np.isclose(float(halo_cosmology[key]), float(expected))}
if mismatches:
    raise ValueError(f'Halo catalog cosmology does not match the AbacusSummit c000 reference: {mismatches}')

print(f'{hoddies_file}')
print(f'{len(halos):,} halos, box {boxsize} Mpc/h, log10 Mh in [{halos["log10_Mh"].min():.2f}, {halos["log10_Mh"].max():.2f}]')

In [ ]:
import multiprocessing
#multiprocessing.cpu_count()
import numba
numba.config.NUMBA_NUM_THREADS
numba.get_num_threads()

## HODDIES

In [ ]:
data_file = np.load('/home/biel/dev-v2/hicola_data/DESI_Y1_LRG/allcounts_LRG_GCcomb_0.4_0.6_default_angular_bitwise_log_njack128_nran4_split20_smu.npy', allow_pickle=True)[()]
#data_file['param']['fit_param']['fit_statistics'] = ['wp', 'xi_ells', 'delta_sigma']

data_file['D1D2'].keys()

In [ ]:
data_file = np.load('/home/biel/dev-v2/hicola_data/DESI_Y1_LRG/allcounts_LRG_GCcomb_0.4_0.6_default_angular_bitwise_log_njack128_nran4_split20_rppi.npy', allow_pickle=True)[()]
#data_file['param']['fit_param']['fit_statistics'] = ['wp', 'xi_ells', 'delta_sigma']

data_file.keys()

In [ ]:
base_catalog = Base_catalogue({name: halos[name] for name in halos.dtype.names},  # a dict: a structured array is refused
                              boxsize=boxsize, z_simu=redshift, nthreads=nthreads)
HOD_obj = HOD(base_catalog, param_file=str(param_file), nthreads=nthreads, seed=seed)
HOD_obj.cosmo = abacus_cosmo  # used for the RSD of get_wp / get_xiells

# Fit results of this run and redshift: dir_output_fit of HODDIES, kept in hicola_data with the
# parameter file and the settings used
fit_dir = hp.hod_fit_dir(target, model, realization, redshift, fit_name)
HOD_obj.args['fit_param']['dir_output_fit'] = str(fit_dir)
fit_dir.mkdir(parents=True, exist_ok=True)
(fit_dir / param_file.name).write_text(param_file.read_text())
import subprocess
hoddies_commit = subprocess.run(['git', '-C', str(hp.HODDIES_ROOT), 'describe', '--always', '--dirty'],
                                capture_output=True, text=True).stdout.strip()
(fit_dir / 'fit_info.json').write_text(json.dumps({
    'run': hp.simulation_name(target, model, realization), 'redshift': redshift, 'boxsize': boxsize,
    'halo_catalog': str(hoddies_file), 'hod_param_file': param_file.name, 'tracer': tracer,
    'nthreads': HOD_obj.nthreads, 'seed': seed, 'hoddies': f'{HODDIES.__file__} ({hoddies_commit})',
}, indent=2))
print(f'Fit results: {fit_dir}')
if HOD_obj.nthreads != nthreads:
    print(f'Warning: HODDIES uses {HOD_obj.nthreads} threads, not {nthreads} (numba thread count)')

print(f'{tracer} HOD parameters:')
print(json.dumps(HOD_obj.args[tracer], indent=1, default=str))

In [ ]:
from pycorr import TwoPointCorrelationFunction

result = TwoPointCorrelationFunction.load('/home/biel/dev-v2/hicola_data/DESI_Y1_LRG/allcounts_LRG_GCcomb_0.4_0.6_default_angular_bitwise_log_njack128_nran4_split20_rppi.npy')[::2][4:-3]
result_smu = TwoPointCorrelationFunction.load('/home/biel/dev-v2/hicola_data/DESI_Y1_LRG/allcounts_LRG_GCcomb_0.4_0.6_default_angular_bitwise_log_njack128_nran4_split20_smu.npy')[16:-6]
HOD_obj.args['clustering_settings']['xi_rppi']['edges_rppi'] = result.edges
HOD_obj.args['clustering_settings']['xi_smu']['edges_smu'] = result_smu.edges

In [ ]:
ells = (0,2)
nells = len(ells)
s, xiell, cov = result_smu.get_corr(ells=ells, return_sep=True, return_cov=True)
std_smu = np.array_split(np.diag(cov)**0.5, nells)
ax = plt.gca()
for ill,ell in enumerate(ells):
    ax.errorbar(s + ill * 0.1, s*xiell[ill], s*std_smu[ill], fmt='-', label=r'$\ell = {:d}$'.format(ell))
ax.legend()
ax.grid(True)
ax.set_xlabel(r'$s$ [Mpc/h]')
ax.set_ylabel(r'$s\xi_{\ell}(s)$')
ax.set_xscale('log')
plt.show()

In [ ]:
pimax = 40
rp, wp, std = result(pimax=pimax, return_sep=True, return_std=True)
ax = plt.gca()
ax.errorbar(rp, rp*wp, rp*std, fmt='-', label=r'$\ell = {:d}$'.format(ell))
ax.legend()
ax.grid(True)
ax.set_xlabel(r'$s$ [Mpc/h]')
ax.set_ylabel(r'$s\xi_{\ell}(s)$')
ax.set_xscale('log')
plt.show()

In [ ]:
data_vec = np.concatenate([wp, xiell[0], xiell[1]])
data_std = np.concatenate([std, std_smu[0], std_smu[1]])
mock_corr = np.loadtxt('/home/biel/dev-v2/hicola_data/DESI_Y1_LRG/M_corr_lrg_z0.5.txt')

mcov = np.outer(data_std, data_std) * mock_corr

In [ ]:
from mpi4py import MPI

HOD_obj.initialize_fit(data_vec, mcov)

mpi_comm = MPI.COMM_WORLD
mpi_rank = mpi_comm.Get_rank()

minimizer_options = {"maxiter":100, "popsize":72, 'xtol':1e-2, 'workers':mpi_comm.Get_size(),  'backend':'mpi'}

res = HOD_obj.run_minimizer(mpi_comm=mpi_comm, seed=1, **minimizer_options)
print(f"Rank {mpi_rank} finished minimization with result: {res}", flush=True)

In [ ]:
mpi_comm.Get_size()


In [ ]:
HOD_obj.nthreads

In [ ]:
dict(zip(HOD_obj.name_params, HOD_obj.result_fit['x']))

In [ ]:
HOD_obj.args['LRG']['M_0'] = HOD_obj.result_fit['x'][0]
HOD_obj.args['LRG']['M_1'] = HOD_obj.result_fit['x'][1]
HOD_obj.args['LRG']['alpha'] = HOD_obj.result_fit['x'][2]
HOD_obj.args['LRG']['f_sigv'] = HOD_obj.result_fit['x'][3]
HOD_obj.args['LRG']['log_Mcent'] = HOD_obj.result_fit['x'][4]
HOD_obj.args['LRG']['sigma_M'] = HOD_obj.result_fit['x'][5]

In [ ]:
cat=HOD_obj.make_mock_cat()

In [ ]:
data_dic = {'wp': (rp, wp, std), 'xi_ells': (s, xiell, std_smu )}
HOD_obj.plot_stats(cat, stats=['wp', 'xi_ells'], data=data_dic)